### Coleta e Categorização de Dados



Checagem de Campos de tabelas ![em](path) um único arquivo

In [0]:
import io
import tempfile
from ftplib import FTP
import pyreaddbc

AGRAVO_TESTE = "ACBI"
ANO_TESTE = "20"
NOME_ARQUIVO = f"{AGRAVO_TESTE}BR{ANO_TESTE}.dbc"

ftp = FTP("ftp.datasus.gov.br", timeout=60)
ftp.login()
achou = False
for diretorio in DIRETORIOS_SINAN:
    ftp.cwd(diretorio)
    if NOME_ARQUIVO.upper() in [n.upper() for n in ftp.nlst()]:
        achou = True
        break
if not achou:
    raise FileNotFoundError(f"{NOME_ARQUIVO} não encontrado em nenhum diretório")

buffer = io.BytesIO()
ftp.retrbinary(f"RETR {NOME_ARQUIVO}", buffer.write)
ftp.quit()

with tempfile.NamedTemporaryFile(delete=False, suffix=".dbc") as tmp:
    caminho_dbc = tmp.name
    tmp.write(buffer.getvalue())
caminho_dbf = caminho_dbc.replace(".dbc", ".dbf")
pyreaddbc.dbc2dbf(caminho_dbc, caminho_dbf)

meta = ler_cabecalho_dbf(caminho_dbf)
nomes_campos = [c["nome"] for c in meta["campos"]]

print(f"{NOME_ARQUIVO}: {meta['n_registros']:,} registros, {len(nomes_campos)} campos\n")
print("Campos:", nomes_campos)

os.remove(caminho_dbc)
os.remove(caminho_dbf)

### Definição de parâmetros

Importando os pacotes necessários

In [0]:
import os
import io
import time
import struct
import tempfile
from ftplib import FTP
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

import numpy as np
import pandas as pd
import pyreaddbc
import gc
import ctypes

Criando função de liberar memória 
Permite a devolução de memória ao sistema operacional depois de cada arquivo processado (dado que há muitos arquivos grandes em sequência, a reserva ocupa memória de forma ociosa)

In [0]:
def liberar_memoria():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

Criação de vetores com agravos, tabelas, anos, colunas e pastas do ftp do SINAN de interesse

Evita carregamento de arquivos desnecessários

Obs. não foi possível incluir Leishmaniose por subcategorização no SINAN

In [0]:
AGRAVOS = [
    "DENGUE", "CHIKUNGUNYA", "ZIKA", "TUBERCULOSE", "MENINGITE",
    "ACIDTRAB", "VIOLENCIA", "HEPATITES", "CANCER", "DERMATOSE", "INTOXICACAO", "LEPTOSPIROSE", "LER/DORT", "MALARIA", "PAIR", "PNEUMOCONIOSE", "TETANO", "TRANSTMENTAL"
]

MAPA_AGRAVOS_FTP = {
    "DENGUE": "DENG", "CHIKUNGUNYA": "CHIK", "ZIKA": "ZIKA", "TUBERCULOSE": "TUBE",
    "MENINGITE": "MENI", "ACIDTRAB": "ACBI", "VIOLENCIA": "VIOL",
    "HEPATITES": "HEPA", "CANCER":"CANC", "DERMATOSE":"DERM", "INTOXICACAO":"IEXO", "LEPTOSPIROSE":"LEPT", "LER/DORT":"LERD", "MALARIA":"MALA", "PAIR":"PAIR", "PNEUMOCONIOSE":"PNEU", "TETANO":"TETA", "TRANSTMENTAL":"TRAC" 
}

ANOS = list(range(2008, date.today().year + 1))

COLUNAS_INTERESSE = [
    "NU_ANO", "DT_NOTIFIC", "ID_MUNICIP", "ID_MN_RESI", "ID_AGRAVO", "CID",
    "CS_SEXO", "CS_RACA", "CAT"
]

DIRETORIOS_SINAN = [
    "/dissemin/publicos/SINAN/DADOS/FINAIS",
    "/dissemin/publicos/SINAN/DADOS/PRELIM",
]

Criação de catálogo, esquema próprio, e volume para armazenamento da tabela final
Obs. Volume pode ser acessado tanto por park quanto por spark

In [0]:
CATALOG = "workspace"
SCHEMA = "sinan"
TABELA = f"{CATALOG}.{SCHEMA}.silver_sinan"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.dados_temp")

Configuração de armazenamento e paralelismo

Define onde cada tipo de dado fica guardado durante a execução (arquivos brutos, baixados do FTP, ficam num diretório temporário comum, e o resultado limpo fica em um Volume separado, para sobreviver entre sessões)

Define também quantas tarefas rodam simultaneamente em cada etapa (6 downloads do FTP do SINAN, mas apenas um arquvio processado por vez, dado que muitos têm milhões de linhas)

In [0]:
OUT_DIR = tempfile.mkdtemp(prefix="sinan_")
RAW_DIR = os.path.join(OUT_DIR, "raw")
PARQUET_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/dados_temp/parquet"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)
print(f"Diretório de trabalho: {OUT_DIR}")

N_THREADS_DOWNLOAD = 6
N_THREADS_PARSE = 1

Definir função de listar arquivos do FTP do SINAN, tanto da pasta FINAL quanto da pasta PRELIM

In [0]:
def listar_arquivos_ftp():
    localizacao = {}
    for diretorio in DIRETORIOS_SINAN:
        ftp = FTP("ftp.datasus.gov.br", timeout=60)
        ftp.login()
        ftp.cwd(diretorio)
        for nome in ftp.nlst():
            localizacao[nome.upper()] = diretorio
        ftp.quit()
    return localizacao

Definir função que separa as tabelas e dados de interesse com o que existe no FTP 

In [0]:
def montar_tarefas(agravos, anos, mapa_ftp, localizacao_arquivos):
    tarefas = []
    for agravo in agravos:
        if agravo not in mapa_ftp:
            print(f"⊘ {agravo}: não mapeado")
            continue
        codigo_ftp = mapa_ftp[agravo]
        for ano in anos:
            ano_str = str(ano)[-2:]
            candidatos = [f"{codigo_ftp}BR{ano_str}.dbc", f"{codigo_ftp}BR{ano_str}.dbf"]
            nome_arquivo = next((c for c in candidatos if c.upper() in localizacao_arquivos), None)
            if nome_arquivo:
                diretorio = localizacao_arquivos[nome_arquivo.upper()]
                tarefas.append({"agravo": agravo, "ano": ano, "nome_arquivo": nome_arquivo, "diretorio": diretorio})
            else:
                print(f"⊘ {agravo} {ano}: não encontrado (FINAIS nem PRELIM)")
    return tarefas


Definir função que executa o download de um único arquivo para o disco local
Essa função roda em paralelo múltiplas vezes

In [0]:
def baixar_arquivo(tarefa: dict) -> dict:
    nome_arquivo = tarefa["nome_arquivo"]
    try:
        ftp = FTP("ftp.datasus.gov.br", timeout=60)
        ftp.login()
        ftp.cwd(tarefa["diretorio"])
        buffer = io.BytesIO()
        ftp.retrbinary(f"RETR {nome_arquivo}", buffer.write)
        ftp.quit()

        caminho = os.path.join(RAW_DIR, nome_arquivo)
        with open(caminho, "wb") as f:
            f.write(buffer.getvalue())

        sufixo = ".dbc" if nome_arquivo.endswith(".dbc") else ".dbf"
        return {**tarefa, "caminho_bruto": caminho, "sufixo": sufixo, "erro": None}
    except Exception as e:
        return {**tarefa, "caminho_bruto": None, "sufixo": None, "erro": str(e)[:150]}

Definir funções que: 
1. Lê só o índice do arquivo DBF (quantos registros, quais colunas), para determinar em qual posição exata (bytes) cada dado começa dentro de cada linha
e
2. Extrai as colunas de interesse do arquivo DBF em uma operação em bloco (sem ser linha por linha)![](path), para ganhar agilidade

In [0]:
def ler_cabecalho_dbf(caminho):
    with open(caminho, "rb") as f:
        cabecalho = f.read(32)
        n_registros = struct.unpack("<I", cabecalho[4:8])[0]
        tam_cabecalho = struct.unpack("<H", cabecalho[8:10])[0]
        tam_registro = struct.unpack("<H", cabecalho[10:12])[0]

        campos = []
        offset = 1
        while True:
            desc = f.read(32)
            if desc[0:1] == b"\x0d":
                break
            nome = desc[0:11].split(b"\x00")[0].decode("ascii", errors="replace")
            tamanho = desc[16]
            campos.append({"nome": nome, "tamanho": tamanho, "offset": offset})
            offset += tamanho

    return {
        "n_registros": n_registros,
        "tam_cabecalho": tam_cabecalho,
        "tam_registro": tam_registro,
        "campos": campos,
    }


def ler_dbf_colunas(caminho, colunas_desejadas, encoding="latin-1"):
    meta = ler_cabecalho_dbf(caminho)
    campos_existentes = {c["nome"]: c for c in meta["campos"]}
    colunas_presentes = [c for c in colunas_desejadas if c in campos_existentes]

    if not colunas_presentes:
        return pd.DataFrame()

    nomes = ["_flag"] + colunas_presentes
    formatos = ["S1"] + [f"S{campos_existentes[c]['tamanho']}" for c in colunas_presentes]
    offsets = [0] + [campos_existentes[c]["offset"] for c in colunas_presentes]

    dtype = np.dtype({
        "names": nomes,
        "formats": formatos,
        "offsets": offsets,
        "itemsize": meta["tam_registro"],
    })

    with open(caminho, "rb") as f:
        f.seek(meta["tam_cabecalho"])
        dados = np.fromfile(f, dtype=dtype, count=meta["n_registros"])

    df = pd.DataFrame(dados)
    df = df.loc[df["_flag"] != b"*", colunas_presentes].reset_index(drop=True)

    for col in colunas_presentes:
        df[col] = df[col].str.decode(encoding, errors="replace").str.strip()

    return df

Definir função que categoriza os dados, padronizando as variáveis dos dados brutos para variáveis com nomes padronizados autoexplicativos, trazendo rótulos para os códigos numéricos

In [0]:
def categorizar(df: pd.DataFrame) -> pd.DataFrame:
    if "NU_ANO" in df.columns:
        df["ANO_NOTIF"] = df["NU_ANO"]
    elif "DT_NOTIFIC" in df.columns:
        df["ANO_NOTIF"] = pd.to_datetime(df["DT_NOTIFIC"], errors="coerce").dt.year

    if "ID_MUNICIP" in df.columns:
        df["COD_MUN_NOTIF"] = df["ID_MUNICIP"].astype(str).str.zfill(6)
        df["COD_UF_NOTIF"] = df["COD_MUN_NOTIF"].str[:2]
        df["COD_REGIAO"] = df["COD_UF_NOTIF"].str[0]
        df["UF_NOTIF"] = df["COD_UF_NOTIF"]

    if "ID_MN_RESI" in df.columns:
        df["COD_MUN_RESI"] = df["ID_MN_RESI"].astype(str).str.zfill(6)

    if "ID_AGRAVO" in df.columns:
        df["CID10"] = df["ID_AGRAVO"].astype(str).str.upper()
        df["CID_CAPITULO"] = df["CID10"].str[0]
    elif "CID" in df.columns:
        df["CID10"] = df["CID"].astype(str).str.upper()
        df["CID_CAPITULO"] = df["CID10"].str[0]

    if "CS_SEXO" in df.columns:
        sexo_map = {"M": "Masculino", "F": "Feminino", "I": "Ignorado"}
        df["SEXO"] = df["CS_SEXO"].map(sexo_map).fillna("Ignorado")
    else:
        df["SEXO"] = "Ignorado"

    if "CS_RACA" in df.columns:
        raca_map = {"1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda", "5": "Indígena", "9": "Ignorado"}
        df["RACA"] = df["CS_RACA"].map(raca_map).fillna("Ignorado")
    else:
        df["RACA"] = "Ignorado"

    cat_map = {"1": "Sim", "2": "Não", "3": "Não se aplica", "9": "Ign/Branco"}
    if "CAT" in df.columns:
        df["EMITIA_CAT"] = df["CAT"].map(cat_map).fillna("Ign/Branco")
    else:
        df["EMITIA_CAT"] = "Não presente"

    colunas = [
        "ANO_NOTIF", "COD_REGIAO", "COD_UF_NOTIF", "UF_NOTIF",
        "COD_MUN_NOTIF", "COD_MUN_RESI", "CID10", "CID_CAPITULO",
        "SEXO", "RACA", "EMITIA_CAT",
    ]
    return df[[c for c in colunas if c in df.columns]].copy()

Definir função "maestro" de processamento — descompactação, leitura vetorizada, categorização, salvamento de resultado como parquet, além de limpar os arquivos temporários finais

In [0]:
def processar_arquivo(download: dict) -> dict:
    if download["erro"] or not download["caminho_bruto"]:
        return {**download, "caminho_parquet": None, "erro_parse": "download falhou"}

    caminho_bruto = download["caminho_bruto"]
    caminho_dbf = caminho_bruto

    try:
        if download["sufixo"] == ".dbc":
            caminho_dbf = caminho_bruto.replace(".dbc", ".dbf")
            pyreaddbc.dbc2dbf(caminho_bruto, caminho_dbf)

        df = ler_dbf_colunas(caminho_dbf, COLUNAS_INTERESSE, encoding="latin-1")

        if df.empty:
            return {**download, "caminho_parquet": None, "erro_parse": "arquivo vazio ou sem colunas de interesse"}

        df_cat = categorizar(df)
        del df
        df_cat["AGRAVO"] = download["agravo"]

        caminho_parquet = os.path.join(PARQUET_DIR, f"{download['agravo']}_{download['ano']}.parquet")
        df_cat.to_parquet(caminho_parquet, index=False)

        return {**download, "caminho_parquet": caminho_parquet, "n_registros": len(df_cat), "erro_parse": None}
    except Exception as e:
        return {**download, "caminho_parquet": None, "erro_parse": str(e)[:150]}
    finally:
        for caminho in {caminho_bruto, caminho_dbf}:
            if caminho and os.path.exists(caminho):
                os.remove(caminho)

### Planejamento e Execução da Coleta em fases
Planejamento por etapas em cascata - que reduzem as tarefas da lista anterior
1. Mapeamento do universo disponível
2. Seleção a partir dos dados de interesse
3. Filtragem da lista a partir do que já existe no Volume

In [0]:
localizacao_arquivos = listar_arquivos_ftp()
print(f"{len(localizacao_arquivos)} arquivos listados no FTP (FINAIS + PRELIM)")

tarefas = montar_tarefas(AGRAVOS, ANOS, MAPA_AGRAVOS_FTP, localizacao_arquivos)
print(f"{len(tarefas)} arquivos a baixar")

tarefas = [
    t for t in tarefas
    if not os.path.exists(os.path.join(PARQUET_DIR, f"{t['agravo']}_{t['ano']}.parquet"))
]
print(f"{len(tarefas)} arquivos ainda faltam (pulando os já processados no Volume)")

Execução em etapas - cada uma com grau de paralelismo
1. Download de tudo em paralelo (mesmo caso falhem)
2. Processamento da fila dos arquivos menores para os maiores, liberando memória após o processamento um a um

In [0]:
t0 = time.perf_counter()
downloads = []
with ThreadPoolExecutor(max_workers=N_THREADS_DOWNLOAD) as executor:
    futures = {executor.submit(baixar_arquivo, t): t for t in tarefas}
    for future in as_completed(futures):
        r = future.result()
        downloads.append(r)
        print(f"⚠ {r['nome_arquivo']}: {r['erro']}" if r["erro"] else f"✓ {r['nome_arquivo']}")
print(f"Downloads: {time.perf_counter() - t0:.1f}s")

downloads_ok = sorted(
    [d for d in downloads if not d["erro"]],
    key=lambda d: os.path.getsize(d["caminho_bruto"]),
)

t0 = time.perf_counter()
resultados = []
with ThreadPoolExecutor(max_workers=N_THREADS_PARSE) as executor:
    futures = {executor.submit(processar_arquivo, d): d for d in downloads_ok}
    for future in as_completed(futures):
        r = future.result()
        resultados.append(r)
        liberar_memoria()
        elapsed = time.perf_counter() - t0
        if r["erro_parse"]:
            print(f"⚠ [{elapsed:.0f}s] {r['agravo']} {r['ano']}: {r['erro_parse']}")
        else:
            print(f"✓ [{elapsed:.0f}s] {r['agravo']} {r['ano']}: {r['n_registros']:,} registros")
print(f"Processamento: {time.perf_counter() - t0:.1f}s")

Consolidação: lê tudo que está no volume, substituindo a tabela definitiva de uma vez

In [0]:
if os.listdir(PARQUET_DIR):
    spark.sql(f"DROP TABLE IF EXISTS {TABELA}")
    spark.read.parquet(PARQUET_DIR).write.mode("overwrite").partitionBy("AGRAVO").saveAsTable(TABELA)
    print(f"Salvo em {TABELA}: {spark.table(TABELA).count():,} registros")
else:
    print("Nenhum dado processado com sucesso")

- LIMPAR! Para quando precisar refazer a camada

In [0]:
#import shutil

#PARQUET_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/dados_temp/parquet"
#for nome in os.listdir(PARQUET_DIR):
 #   os.remove(os.path.join(PARQUET_DIR, nome))
#print(f"Volume limpo: {PARQUET_DIR}")